In [ ]:
# NLP THEORY NOTE — Pipeline Overview
# This notebook implements an NLP pipeline that can be related to several
# core NLP concepts:
#
# 1. OCR (EasyOCR) is NOT an NLP task — it is computer vision. It converts
#    pixel patterns into character sequences. NLP begins only after OCR
#    produces text.
#
# 2. The pipeline exemplifies a key NLP distinction: text vectorization vs.
#    word embedding. The sentence transformer used here (paraphrase-
#    multilingual-MiniLM-L12-v2) belongs to the embedding family — it maps
#    text into a dense, low-dimensional vector space where semantic
#    similarity corresponds to geometric proximity. This is fundamentally
#    different from simpler text vectorization approaches (bag-of-words,
#    TF-IDF) which encode frequency statistics but no semantic relationships.
#
# 3. The pipeline is presented as an NLP-based alternative to VLM-based
#    image classification. The key distinction: a VLM (e.g. MiniCPM) processes
#    text as visual ink-on-page pixel patterns; this pipeline treats extracted
#    text as linguistic objects with semantic content. These constitute
#    genuinely independent sources of evidence.

In [ ]:
import os
from pathlib import Path
#from ultralytics import YOLO
from PIL import Image
import shutil
import pandas as pd
from source import image_id_converter as img_idc
#from source import sort_img_files as sif
import matplotlib.pyplot as plt
from source import llm_input as llm_i
from source import llm_output as llm_o
import os
from PIL import Image

In [ ]:
import ollama
import json
import re
import pickle

In [ ]:
os.getcwd()

# Using LLM (mini-CPM) for image analysis

## Define Functions:

In [ ]:
def add_pred_values(idx, labels_results, columns, values_to_add):
    selection_bools = labels_results.image_id == idx
    
    labels_results.loc[selection_bools, columns] = values_to_add

In [ ]:
# def analyse_giub_img_dir_llm(jpg_data_path, create_analysis_prompt, model_function):
#     # Get time stamp:
#     timestamp_start_is_photo_analysis = pd.Timestamp.now()
#     
#     # Get list of image files to analyse: 
#     image_files = os.listdir(jpg_data_path)
#     img_ids = [image_file.split('Oberland')[1].split('.')[0] for image_file in image_files]
#     
#     # Make empty dictionary to store results:
#     image_descr = {}
#     
#     # Loop through images to get answers: 
#     for image_file in image_files:
#         image_path = jpg_data_path / image_file
#         path_str = str(image_path)
#         #print('\n')
#         #print(path_str)
#         parts = path_str.split('.jpg')
#         img_id = parts[-2][-3:]
#     
#         # Analyse image, get values for each of the categorical variables specified above:
#         #image_description = analyze_image_structured(image_path)
#         #image_description = llm_o.analyze_image_structured(image_path, create_analysis_prompt)
#         image_description = llm_o.analyze_image_structured(image_path, create_analysis_prompt, model_function)
#         
#         dict_type_bool = type(image_description) == dict
#             
#         #print(image_description)
#         image_descr[img_id] = image_description
#     
#     timestamp_end_is_photo_analysis = pd.Timestamp.now()
# 
#     return timestamp_start_is_photo_analysis, timestamp_end_is_photo_analysis, image_descr
#     

In [ ]:
def store_duration(time_analysis_dict, time_analysis_for_df_dict, analysis_name, duration,
                  timestamp_start_is_photo_analysis,
                  timestamp_end_is_photo_analysis):
    time_analysis_dict[analysis_name] = {}
    time_analysis_dict[analysis_name]['duration_str'] = f"Analysis took: {duration}"
    time_analysis_dict[analysis_name]['duration_seconds'] = total_seconds
    time_analysis_dict[analysis_name]['duration_seconds_str'] = f"Analysis took: {total_seconds:.2f} seconds"
    time_analysis_dict[analysis_name]['duration_minutes'] = total_seconds/60
    time_analysis_dict[analysis_name]['duration_minutes_str'] = f"Analysis took: {total_seconds/60:.2f} minutes"
    time_analysis_dict[analysis_name]['time_stamp_start'] = timestamp_start_is_photo_analysis
    time_analysis_dict[analysis_name]['time_stamp_end'] = timestamp_end_is_photo_analysis

    time_analysis_for_df_dict['analysis_name'].append(analysis_name)
    time_analysis_for_df_dict['time_stamp_start'].append(timestamp_start_is_photo_analysis)
    time_analysis_for_df_dict['duration_str'].append(f"Analysis took: {duration}")
    time_analysis_for_df_dict['duration_seconds'].append(total_seconds)
    time_analysis_for_df_dict['duration_seconds_str'].append(f"Analysis took: {total_seconds:.2f} seconds")
    time_analysis_for_df_dict['duration_minutes'].append(total_seconds/60)
    time_analysis_for_df_dict['duration_minutes_str'].append(f"Analysis took: {total_seconds/60:.2f} minutes")

    return time_analysis_dict, time_analysis_for_df_dict

### Prepare empty dictionary for time analyses and get time stamp for overall workflow duration:

In [ ]:
time_analyses = {}
time_analyses_for_df = {}
time_analyses_for_df['analysis_name'] = []
time_analyses_for_df['time_stamp_start'] = []
time_analyses_for_df['duration_str'] = []
time_analyses_for_df['duration_seconds'] = []
time_analyses_for_df['duration_seconds_str'] = []
time_analyses_for_df['duration_minutes'] = []
time_analyses_for_df['duration_minutes_str'] = []

timestamp_start_workflow = pd.Timestamp.now()
timestamp_start_workflow

### Define LLM model to be used:

In [ ]:
#model_function = llm_i.call_minicpm_model


## Set paths:

In [ ]:
#root_path = Path('/Users/stephanehess/Documents/CAS_AML/dias_digit_project')
#root_path = Path('/Users/stephanehess/Documents/CAS_AML/dias_digit_project/test_yolo_object_train')

project_path = Path.cwd()
#root_path = project_path
root_path = (project_path / '..' / 'data_folders').resolve()
#root_path = (project_path / '..' / 'test_yolo_object_train').resolve()
#root_path = project_path / 'test_llm_img_analysis'
data_path = root_path / 'data'
tif_data_path = root_path / 'data_1'
#data_path = root_path / 'visual_genome_data_all'
jpg_data_path = root_path / 'data_jpg'
#yolo_path = root_path / 'visual_genome_yolo_all'
output_dir_not_photo = root_path / 'not_photo'
output_dir_with_person = root_path / 'with_person'
output_dir_without_person = root_path / 'without_person'



### Copy and convert image files from tif_data_path to jpg_data_path:

In [ ]:

source_folder = tif_data_path
destination_folder = jpg_data_path

llm_i.convert_tif_to_jpg(source_folder, destination_folder, quality=100)


## Create directories for sorting the images:

In [ ]:
# Create output directories
#os.chdir(root_path/'..')
os.makedirs(output_dir_not_photo, exist_ok=True)
os.makedirs(output_dir_with_person, exist_ok=True)
os.makedirs(output_dir_without_person, exist_ok=True)
#os.chdir('root_path')

### Load label data (ground truth) to compare to LLM responses:

The file with_without_person.csv contains labels added by (human) visual inspection that represent the ground truth. 
 * Column with_person: whether or not any person is in the image.
 * Column recognisable: whether any person that would be recognisable to a human familiar with said person is in the image.
 * Column photo: whether or not the image is a photograph (as opposed to some other kind of representation such as map, drawing, painting, scheme, figure)
 * Column church: whether or not any church is in the image.
 * Column high_alpine_environment: whether or not the scene shown in the image is situated in a high alpine environment (according to non-expert human judgement)

In [ ]:
label_data = pd.read_csv(data_path/'labels_mod.csv')
label_data.head()

In [ ]:
img_ids = list(label_data.image_id)

In [ ]:
# Reconvert image ids to integers (e.g. '234') as strings from the form they were saved in (e.g. 'id234' to ensure 
# string data type to deal with duck typing): 
label_data['image_id'] = img_idc.reconvert_image_ids(img_ids)

In [ ]:
label_data.head()

In [ ]:

# Make sure no .DS_Store file is included in jpg_data_path: 
import os
ds_file_path = jpg_data_path / '.DS_Store'

# Remove a specific .DS_Store file
if os.path.exists(ds_file_path):
    os.remove(ds_file_path)
    print("Removed .DS_Store")
else:
    print(".DS_Store not found")

# Find all .ipynb_checkpoints directories
for checkpoint_dir in jpg_data_path.rglob('.ipynb_checkpoints'):
    if checkpoint_dir.is_dir():
        print(f"Removing: {checkpoint_dir}")
        shutil.rmtree(checkpoint_dir)



# Get list of image files present:
image_files = os.listdir(jpg_data_path)

#image_files.remove(".ipynb_checkpoints")



# Extract image ids from image file names:
img_ids = [image_file.split('Oberland')[1].split('.')[0] for image_file in image_files]
img_ids.sort()
print(img_ids)

# Select relevant rows from label_data data frame by id list: 
select_bools = [img_id in img_ids for img_id in label_data.image_id]

label_data = label_data[select_bools].copy()
label_data

### Keep Code until here!

In [ ]:
# --- New imports for OCR + NLP pipeline ---
import easyocr
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim
from langdetect import detect, DetectorFactory
import numpy as np

# Fix random seed in langdetect for reproducibility
DetectorFactory.seed = 0

In [ ]:
# --- Initialise OCR reader and sentence transformer model ---

# EasyOCR reader: supports German, French, Italian and English
# gpu=False ensures it runs on CPU (safe for M1 Mac)
print("Loading OCR reader...")
ocr_reader = easyocr.Reader(['de', 'fr', 'it', 'en'], gpu=False)
print("OCR reader ready.")

# Multilingual sentence transformer model
# This model handles German, French, Italian and English well
# and is small enough to run comfortably on a laptop
print("Loading sentence transformer model...")
sentence_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print("Sentence transformer model ready.")

In [ ]:
# NLP THEORY NOTE — OCR and Tokenization
# EasyOCR performs implicit tokenization: it segments the image into
# character and word units. This is analogous to the first step in any
# NLP pipeline. Note that EasyOCR supports multiple languages (de, fr, it, en)
# simultaneously — this is non-trivial because word boundary detection
# differs across languages. Compare this with the BPE (Byte Pair Encoding)
# tokenization used in transformer models like GPT, which avoids language-
# specific word boundary assumptions entirely by starting from individual
# characters and merging frequent pairs iteratively.

In [ ]:
# --- Define core pipeline functions ---

def run_ocr(image_path, reader):
    """
    Run OCR on a single image.
    Returns:
        - full_text (str): all detected text joined into one string
        - raw_results (list): raw EasyOCR output [(bbox, text, confidence), ...]
    """
    raw_results = reader.readtext(str(image_path))
    # Only keep results above a minimum confidence threshold
    texts = [text for (_, text, conf) in raw_results if conf > 0.3]
    full_text = ' '.join(texts)
    return full_text, raw_results




In [ ]:
# NLP THEORY NOTE — Language Identification
# detect_language() uses langdetect, which is a classic NLP task: identifying
# the language of a text from its statistical character and word patterns.
# Note the reliability limitation flagged in the results: language detection
# on very short texts (< ~50 characters) is unreliable. This illustrates a
# general principle in NLP — most models require sufficient context to make
# meaningful inferences. Single words or short fragments do not carry enough
# linguistic signal. This is directly analogous to the problem that motivated
# contextual word embeddings (ELMo, BERT): a single word like "bank" has no
# deterministic meaning without surrounding context.

In [ ]:
def detect_language(text):
    """
    Detect the language of extracted text.
    Returns a language code (e.g. 'de', 'fr', 'it') or 'unknown'.
    """
    if not text.strip():
        return 'unknown'
    try:
        return detect(text)
    except Exception:
        return 'unknown'



In [ ]:
# NLP THEORY NOTE — Sentence Transformers and Embedding Similarity
# classify_text() uses cosine similarity between embeddings — this is the
# operational heart of the NLP pipeline and deserves careful explanation:
#
# The sentence transformer (paraphrase-multilingual-MiniLM-L12-v2) is a
# transformer-based model (related to BERT architecture) that encodes an
# entire text passage into a single dense vector in a high-dimensional
# semantic space. Unlike word2vec or fastText which produce static per-word
# embeddings, sentence transformers produce a single vector for the entire
# input, capturing contextual meaning across the full passage.
#
# Cosine similarity measures the angle between two vectors in this space —
# not their Euclidean distance. Two vectors pointing in the same direction
# (angle ≈ 0°, cosine ≈ 1.0) are semantically similar; perpendicular vectors
# (cosine ≈ 0) are unrelated. This is zero-shot classification: no labelled
# training examples are needed. The model was pre-trained on large multilingual
# corpora and the learned geometric structure of its embedding space is what
# makes unseen label comparisons meaningful.
#
# This is NOT keyword matching — the model can recognise that "Deckenschotter"
# (a geological term) belongs near "Alpine geology" in embedding space without
# ever having seen this specific pairing, because it learned the underlying
# semantic structure of German geological vocabulary during pre-training.
#
# IMPORTANT CAVEAT: This is also not a transformer doing what we discussed in
# the "Attention is All You Need" chapter. The sentence transformer here is
# used purely as a frozen feature extractor — its weights are fixed after
# pre-training. We are not training a new model; we are exploiting the
# geometric structure of a pre-learned embedding space.

In [ ]:
def classify_text(text, candidate_labels, model):
    """
    Classify text by computing cosine similarity between the text embedding
    and each candidate label embedding.
    Returns a dict of {label: similarity_score} sorted highest to lowest.
    """
    if not text.strip():
        return {label: 0.0 for label in candidate_labels}
    
    text_embedding = model.encode(text, convert_to_tensor=True)
    label_embeddings = model.encode(candidate_labels, convert_to_tensor=True)
    similarities = cos_sim(text_embedding, label_embeddings)[0]
    
    results = {label: float(sim) for label, sim in zip(candidate_labels, similarities)}
    return dict(sorted(results.items(), key=lambda x: x[1], reverse=True))

In [ ]:
# --- Define candidate labels for classification ---

# Image type labels: what kind of image is this?
# These are written as short descriptive phrases rather than single words
# because sentence transformers work better with phrases that carry more meaning
image_type_labels = [
    'geographical map showing place names and terrain',
    'geological cross-section or schematic profile of rock layers',
    'statistical chart or diagram with numerical data'
]

# Semantic concept labels: what domain does the text belong to?
# These help enrich the classification with additional context
concept_labels = [
    'Alpine geology and rock formations',
    'glaciology and ice',
    'cartography and topography',
    'hydrology and water',
    'statistics and data',
    'human settlement and infrastructure',
]

# Language region labels: useful for geographic attribution
# Note: we also use langdetect separately, but this gives a
# semantics-based cross-check via the extracted text content
language_region_labels = [
    'German-speaking region of Switzerland',
    'French-speaking region of Switzerland',
    'Italian-speaking region of Switzerland',
]

print("Labels defined:")
print(f"  Image types: {len(image_type_labels)}")
print(f"  Concepts: {len(concept_labels)}")
print(f"  Language regions: {len(language_region_labels)}")

In [ ]:
# NLP THEORY NOTE — The Full Pipeline in Context
# The loop below implements a three-stage NLP pipeline:
#
# Stage 1 — OCR (computer vision, not NLP):
#   Raw pixels → character sequences
#
# Stage 2 — Language detection (classical NLP):
#   Character sequences → language label
#   This is a well-established NLP task, here performed by langdetect
#   using statistical n-gram models.
#
# Stage 3 — Semantic classification (modern NLP, embedding-based):
#   Text → dense vector (via sentence transformer) → similarity scores
#   vs. candidate label vectors → predicted category
#
# Note what this pipeline does NOT do that a full NLP pipeline might:
# - Named Entity Recognition (NER): identifying and classifying place names,
#   person names, organisations. NER would be the appropriate tool for
#   extracting Swiss canton names from map labels and attributing images
#   to geographic regions — the original motivation for this project.
# - Parsing: analysing grammatical structure of extracted sentences.
# - Coreference resolution: linking pronouns to their referents.
# The pipeline is deliberately kept lightweight given the task requirements
# and time constraints — but NER would be the natural next component to add.

In [ ]:
# --- Main analysis loop ---
# For each image: run OCR, detect language, classify image type and concepts

timestamp_start_ocr_nlp = pd.Timestamp.now()

# Empty dictionary to store raw results per image
image_descr = {}

for image_file in image_files:
    image_path = jpg_data_path / image_file
    
    # Extract image id from filename (e.g. 'BernerOberland042.jpg' -> '042')
    img_id = image_file.split('Oberland')[1].split('.')[0]
    
    print(f"Processing image {img_id}...", end=' ')
    
    # Step 1: OCR
    full_text, raw_ocr = run_ocr(image_path, ocr_reader)
    
    # Step 2: Language detection
    detected_language = detect_language(full_text)
    
    # Step 3: Image type classification
    image_type_scores = classify_text(full_text, image_type_labels, sentence_model)
    
    # Step 4: Concept classification
    concept_scores = classify_text(full_text, concept_labels, sentence_model)
    
    # Step 5: Language region classification
    region_scores = classify_text(full_text, language_region_labels, sentence_model)
    
    # Store all results for this image
    image_descr[img_id] = {
        'image_id': img_id,
        'ocr_text': full_text,
        'ocr_raw': raw_ocr,
        'detected_language': detected_language,
        'image_type_scores': image_type_scores,
        'concept_scores': concept_scores,
        'region_scores': region_scores,
    }
    
    print(f"done. Language: {detected_language} | Text length: {len(full_text)} chars")

timestamp_end_ocr_nlp = pd.Timestamp.now()
print(f"\nAll images processed.")
print(f"Duration: {timestamp_end_ocr_nlp - timestamp_start_ocr_nlp}")

In [ ]:
# NLP THEORY NOTE — Interpreting the Results
# The summary table reveals an important empirical finding with theoretical
# implications: classification confidence (cosine similarity scores) correlates
# strongly with OCR text length. Images with < ~50 characters of extracted
# text produce unreliable classifications. This is consistent with the
# theoretical principle that semantic embeddings require sufficient linguistic
# context to be informative.
#
# Concretely: the sentence transformer embeds the ENTIRE extracted text as
# a single vector. A single word like "Profil" produces a vector that could
# reasonably match multiple categories. A longer passage like "Profil vallée
# du Rhône" produces a vector that more specifically points toward geological
# cross-section + French-speaking region. More context = more discriminative
# embedding. This mirrors the motivation for contextual embeddings over static
# ones: meaning emerges from context, not from isolated tokens.
#
# The concept classification (cartography, Alpine geology, hydrology) proved
# more reliable than image type classification (map vs. schematic vs. diagram).
# A possible explanation: concept categories are more semantically distinct
# in the embedding space the model was trained on, whereas the distinction
# between "map" and "geological schematic" may rely on visual rather than
# linguistic cues — reinforcing the argument that text-based and vision-based
# classification constitute genuinely independent methods.

In [ ]:
# --- Inspect individual results ---

def print_image_result(img_id, image_descr):
    """Print a readable summary of the pipeline results for one image."""
    result = image_descr[img_id]
    print(f"=== Image {img_id} ===")
    print(f"OCR text ({len(result['ocr_text'])} chars):")
    print(f"  '{result['ocr_text'][:300]}{'...' if len(result['ocr_text']) > 300 else ''}'")
    print(f"Detected language: {result['detected_language']}")
    print(f"Image type scores:")
    for label, score in result['image_type_scores'].items():
        print(f"  {score:.3f} | {label}")
    print(f"Top concept:")
    top_concept = list(result['concept_scores'].items())[0]
    print(f"  {top_concept[1]:.3f} | {top_concept[0]}")
    print(f"Top language region:")
    top_region = list(result['region_scores'].items())[0]
    print(f"  {top_region[1]:.3f} | {top_region[0]}")
    print()

# Inspect the most text-rich images
for img_id in ['083', '084', '074', '078', '091', '080']:
    print_image_result(img_id, image_descr)

In [ ]:
# --- Build summary dataframe of results ---

def get_top_label(scores_dict):
    """Return the label with the highest similarity score."""
    return max(scores_dict, key=scores_dict.get)

def get_top_score(scores_dict):
    """Return the highest similarity score."""
    return max(scores_dict.values())

# Build a summary row for each image
summary_rows = []

for img_id, result in image_descr.items():
    
    top_image_type = get_top_label(result['image_type_scores'])
    top_image_type_score = get_top_score(result['image_type_scores'])
    
    top_concept = get_top_label(result['concept_scores'])
    top_concept_score = get_top_score(result['concept_scores'])
    
    top_region = get_top_label(result['region_scores'])
    top_region_score = get_top_score(result['region_scores'])
    
    summary_rows.append({
        'image_id': img_id,
        'ocr_text_length': len(result['ocr_text']),
        'detected_language': result['detected_language'],
        'top_image_type': top_image_type,
        'top_image_type_score': round(top_image_type_score, 3),
        'top_concept': top_concept,
        'top_concept_score': round(top_concept_score, 3),
        'top_region': top_region,
        'top_region_score': round(top_region_score, 3),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df = summary_df.sort_values('ocr_text_length', ascending=False).reset_index(drop=True)

# Show images that had text extracted (the interesting ones)
summary_df[summary_df['ocr_text_length'] > 0]

In [ ]:
# --- Build results_tabular structure mirroring existing output format ---

# Generate timestamp id for this analysis run
timestamp_id_nlp = timestamp_start_ocr_nlp.strftime('%Y%m%d_%H%M%S')

# Define a minimum text length below which we consider results unreliable
MIN_TEXT_LENGTH = 50

# --- Build predictions DataFrames (one per classification category) ---
# This mirrors the structure: results_tabular[timestamp_id]['predictions'][category]

# 1. Image type prediction
image_type_rows = []
for img_id, result in image_descr.items():
    top_type = get_top_label(result['image_type_scores'])
    top_score = get_top_score(result['image_type_scores'])
    text_length = len(result['ocr_text'])
    image_type_rows.append({
        'image_id': img_id,
        'image_type_pred': top_type,
        'image_type_score': round(top_score, 3),
        'reliable': text_length >= MIN_TEXT_LENGTH
    })
df_image_type = pd.DataFrame(image_type_rows).sort_values('image_id').reset_index(drop=True)

# 2. Concept prediction
concept_rows = []
for img_id, result in image_descr.items():
    top_concept = get_top_label(result['concept_scores'])
    top_score = get_top_score(result['concept_scores'])
    text_length = len(result['ocr_text'])
    concept_rows.append({
        'image_id': img_id,
        'concept_pred': top_concept,
        'concept_score': round(top_score, 3),
        'reliable': text_length >= MIN_TEXT_LENGTH
    })
df_concept = pd.DataFrame(concept_rows).sort_values('image_id').reset_index(drop=True)

# 3. Language region prediction
region_rows = []
for img_id, result in image_descr.items():
    top_region = get_top_label(result['region_scores'])
    top_score = get_top_score(result['region_scores'])
    text_length = len(result['ocr_text'])
    region_rows.append({
        'image_id': img_id,
        'region_pred': top_region,
        'region_score': round(top_score, 3),
        'reliable': text_length >= MIN_TEXT_LENGTH
    })
df_region = pd.DataFrame(region_rows).sort_values('image_id').reset_index(drop=True)

# 4. Language detection (from langdetect, not sentence transformer)
language_rows = []
for img_id, result in image_descr.items():
    text_length = len(result['ocr_text'])
    language_rows.append({
        'image_id': img_id,
        'detected_language': result['detected_language'],
        'reliable': text_length >= MIN_TEXT_LENGTH
    })
df_language = pd.DataFrame(language_rows).sort_values('image_id').reset_index(drop=True)

# --- Assemble results_tabular mirroring existing structure ---
results_tabular = {}
results_tabular[timestamp_id_nlp] = {
    'pipeline_id': 'ocr_sentencetransformer_v1',
    'pipeline_name': 'ocr_sentencetransformer',
    'pipeline_description': (
        'Pipeline: EasyOCR text extraction -> langdetect language detection -> '
        'sentence-transformers (paraphrase-multilingual-MiniLM-L12-v2) embedding similarity '
        'classification for image type, semantic concept, and language region.'
    ),
    'predictions': {
        'image_type': df_image_type,
        'concept': df_concept,
        'region': df_region,
        'language': df_language,
    }
}

# Quick check
print("Results structure built.")
print(f"Timestamp id: {timestamp_id_nlp}")
print(f"Prediction categories: {list(results_tabular[timestamp_id_nlp]['predictions'].keys())}")
print(f"\nSample - image type predictions:")
print(df_image_type[df_image_type['reliable']].head(10))

In [ ]:
timestamp_id_nlp

In [ ]:
# --- Save all outputs to disk ---

# 1. Save raw image descriptions (OCR text, scores, etc.)
filename_descr = 'responses_ocr_nlp_' + timestamp_id_nlp + '.pkl'
path_descr = data_path / filename_descr
with open(path_descr, 'wb') as f:
    pickle.dump(image_descr, f)

# Reload and verify
with open(path_descr, 'rb') as f:
    reloaded_image_descr = pickle.load(f)
print(f"image_descr saved and verified: {len(image_descr) == len(reloaded_image_descr)}")


# 2. Save results_tabular (structured predictions)
filename_results = 'results_ocr_nlp_' + timestamp_id_nlp + '.pkl'
path_results = data_path / filename_results
with open(path_results, 'wb') as f:
    pickle.dump(results_tabular, f)

# Reload and verify
with open(path_results, 'rb') as f:
    reloaded_results_tabular = pickle.load(f)
print(f"results_tabular saved and verified: {results_tabular.keys() == reloaded_results_tabular.keys()}")


# 3. Save summary dataframe as CSV (useful for quick inspection)
filename_summary = 'summary_ocr_nlp_' + timestamp_id_nlp + '.csv'
path_summary = data_path / filename_summary
summary_df.to_csv(path_summary, index=False)

# Reload and verify
reloaded_summary = pd.read_csv(path_summary)
print(f"summary_df saved and verified: {len(summary_df) == len(reloaded_summary)}")


# 4. Save timing information
duration = timestamp_end_ocr_nlp - timestamp_start_ocr_nlp
total_seconds = duration.total_seconds()

time_analyses['ocr_nlp_pipeline'] = {
    'duration_str': f"Analysis took: {duration}",
    'duration_seconds': total_seconds,
    'duration_seconds_str': f"Analysis took: {total_seconds:.2f} seconds",
    'duration_minutes': total_seconds / 60,
    'duration_minutes_str': f"Analysis took: {total_seconds / 60:.2f} minutes",
    'time_stamp_start': timestamp_start_ocr_nlp,
    'time_stamp_end': timestamp_end_ocr_nlp,
}

filename_times = 'times_ocr_nlp_' + timestamp_id_nlp + '.pkl'
path_times = data_path / filename_times
with open(path_times, 'wb') as f:
    pickle.dump(time_analyses, f)

# Reload and verify
with open(path_times, 'rb') as f:
    reloaded_times = pickle.load(f)
print(f"time_analyses saved and verified: {time_analyses.keys() == reloaded_times.keys()}")

print(f"\nAll files saved to: {data_path}")
print(f"Files saved:")
print(f"  {filename_descr}")
print(f"  {filename_results}")
print(f"  {filename_summary}")
print(f"  {filename_times}")

In [ ]:
# --- Final wrap-up ---

timestamp_end_workflow = pd.Timestamp.now()
total_workflow_duration = timestamp_end_workflow - timestamp_start_workflow
total_workflow_seconds = total_workflow_duration.total_seconds()

print("=" * 55)
print("OCR + NLP PIPELINE COMPLETE")
print("=" * 55)
print(f"Workflow start:    {timestamp_start_workflow}")
print(f"Workflow end:      {timestamp_end_workflow}")
print(f"Total duration:    {total_workflow_seconds/60:.2f} minutes")
print()
print("DATASET SUMMARY")
print("-" * 55)
print(f"Total images processed:  {len(image_descr)}")
print(f"Images with OCR text:    {sum(1 for r in image_descr.values() if len(r['ocr_text']) > 0)}")
print(f"Images with reliable text (>= {MIN_TEXT_LENGTH} chars): "
      f"{sum(1 for r in image_descr.values() if len(r['ocr_text']) >= MIN_TEXT_LENGTH)}")
print()
print("LANGUAGE DETECTION (reliable images only)")
print("-" * 55)
reliable = {k: v for k, v in image_descr.items() if len(v['ocr_text']) >= MIN_TEXT_LENGTH}
lang_counts = pd.Series([v['detected_language'] for v in reliable.values()]).value_counts()
print(lang_counts.to_string())
print()
print("IMAGE TYPE PREDICTIONS (reliable images only)")
print("-" * 55)
type_counts = df_image_type[df_image_type['reliable']]['image_type_pred'].value_counts()
print(type_counts.to_string())
print()
print("TOP CONCEPTS (reliable images only)")
print("-" * 55)
concept_counts = df_concept[df_concept['reliable']]['concept_pred'].value_counts()
print(concept_counts.to_string())
print()
print("OUTPUT FILES")
print("-" * 55)
print(f"  {filename_descr}")
print(f"  {filename_results}")
print(f"  {filename_summary}")
print(f"  {filename_times}")

In [ ]:
# --- Update time_analyses_for_df to match existing timing structure ---

time_analyses_for_df['analysis_name'].append('ocr_nlp_pipeline')
time_analyses_for_df['time_stamp_start'].append(timestamp_start_ocr_nlp)
time_analyses_for_df['duration_str'].append(f"Analysis took: {duration}")
time_analyses_for_df['duration_seconds'].append(total_seconds)
time_analyses_for_df['duration_seconds_str'].append(f"Analysis took: {total_seconds:.2f} seconds")
time_analyses_for_df['duration_minutes'].append(total_seconds / 60)
time_analyses_for_df['duration_minutes_str'].append(f"Analysis took: {total_seconds / 60:.2f} minutes")

# Save updated time_analyses_for_df
filename_times_df = 'times_df_ocr_nlp_' + timestamp_id_nlp + '.pkl'
path_times_df = data_path / filename_times_df
with open(path_times_df, 'wb') as f:
    pickle.dump(time_analyses_for_df, f)

# Reload and verify
with open(path_times_df, 'rb') as f:
    reloaded_times_df = pickle.load(f)
print(f"time_analyses_for_df saved and verified: {time_analyses_for_df.keys() == reloaded_times_df.keys()}")

# Display as data

In [ ]:
MIN_TEXT_LENGTH

In [ ]:
def plot_image_grid(image_ids, jpg_data_path, title, max_images=110, figsize=(15, 10)):
    """
    Plot a grid of images with their image_id as title.
    Shows at most max_images images.
    """
    ids_to_plot = image_ids[:max_images]
    n = len(ids_to_plot)
    if n == 0:
        print(f"No images to plot for: {title}")
        return
    
    ncols = 4
    nrows = math.ceil(n / ncols)
    
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    
    # Always flatten to 1D array regardless of shape
    axes = np.array(axes).flatten()
    
    for i, img_id in enumerate(ids_to_plot):
        # Find the corresponding filename
        matching = [f for f in image_files if f'Oberland{img_id}' in f]
        if not matching:
            axes[i].axis('off')
            continue
        image_path = jpg_data_path / matching[0]
        img = Image.open(image_path)
        axes[i].imshow(img)
        axes[i].set_title(f"id: {img_id}", fontsize=8)
        axes[i].axis('off')
    
    # Hide unused subplots
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')
    
    fig.suptitle(title, fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()


# --- Plot reliable images grouped by top concept ---
reliable_df = summary_df[summary_df['ocr_text_length'] >= MIN_TEXT_LENGTH].copy()

for concept in reliable_df['top_concept'].unique():
    ids = reliable_df[reliable_df['top_concept'] == concept]['image_id'].tolist()
    plot_image_grid(ids, jpg_data_path,
                    title=f"Top concept: {concept} (n={len(ids)})")

# --- Plot a sample of no-text images ---
no_text_ids = summary_df[summary_df['ocr_text_length'] == 0]['image_id'].tolist()
import random
random.seed(42)
sample_no_text = random.sample(no_text_ids, min(20, len(no_text_ids)))
plot_image_grid(sample_no_text, jpg_data_path,
                title=f"Sample of images with no OCR text (n={len(no_text_ids)} total)")

In [ ]:
sample_no_text

In [ ]:
summary_df.shape

In [ ]:
summary_df.head()

### Check images with few words:

In [ ]:

word_counts = list(range(1,700))

sel_bools = [(0<word_counts.count(x)) for x in summary_df.ocr_text_length]
print(sel_bools[0:3])
sum(sel_bools)

In [ ]:
summary_df[sel_bools].shape

In [ ]:
summary_df[sel_bools]
little_text_ids = summary_df[sel_bools]['image_id'].tolist()
plot_image_grid(little_text_ids, jpg_data_path,
                title=f"Sample of images with little OCR text (n={len(no_text_ids)} total)")